Import packages

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import altair as alt
import pickle
import gdown
import matplotlib.pyplot as plt
import folium
from geopy.geocoders import Nominatim
import openrouteservice
import time
import seaborn as sns
import openpyxl
import scipy.stats as stats
import requests
from pandas_datareader import wb

Get history sales file and reduce memory use

In [2]:
sales_history = pd.read_parquet("C:/Users/m.muller/Documents/Python/Projects/Supermarket CFGS/Data/combined_history_items_stores.parquet")

for col in sales_history.select_dtypes(include=["object"]).columns:
    sales_history[col] = sales_history[col].astype("category")

for col in sales_history.select_dtypes(include=["number"]).columns:
    sales_history[col] = pd.to_numeric(sales_history[col], downcast="integer")
    sales_history[col] = pd.to_numeric(sales_history[col], downcast="float")

print(sales_history.head())
print()
print(sales_history.shape)

   id  store_nbr  item_nbr  unit_sales  onpromotion  day    month  \
0   0         25  103665.0         7.0         <NA>    1  2013-01   
1   1         25  105574.0         1.0         <NA>    1  2013-01   
2   2         25  105575.0         2.0         <NA>    1  2013-01   
3   3         25  108079.0         1.0         <NA>    1  2013-01   
4   4         25  108701.0         1.0         <NA>    1  2013-01   

         family  class  perishable     city        state type  cluster  
0  BREAD/BAKERY   2712           1  Salinas  Santa Elena    D        1  
1     GROCERY I   1045           0  Salinas  Santa Elena    D        1  
2     GROCERY I   1045           0  Salinas  Santa Elena    D        1  
3     GROCERY I   1030           0  Salinas  Santa Elena    D        1  
4          DELI   2644           1  Salinas  Santa Elena    D        1  

(125497040, 14)


Filter sales_history on family 'dairy' and store_nbr '44, 45, 47, 49, 46, 48'

In [3]:
dairy_sales_filtered = sales_history[
    (sales_history['family'] == 'DAIRY') &
    (sales_history['store_nbr'].isin([44, 45, 46, 47, 48, 49, 50 ,51]))
]

#index opnieuw instellen
dairy_sales_filtered = dairy_sales_filtered.reset_index(drop=True)

#add column date (yyyy-mm-dd)
dairy_sales_filtered['date'] = pd.to_datetime(
    dairy_sales_filtered['month'].astype(str) + '-' + dairy_sales_filtered['day'].astype(str).str.zfill(2)
)

print(dairy_sales_filtered.head())
print()
print(dairy_sales_filtered.shape)


      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0        37.0         <NA>    2  2013-01  DAIRY   
4  32370         44  164088.0         8.0         <NA>    2  2013-01  DAIRY   

   class  perishable   city      state type  cluster       date  
0   2124           1  Quito  Pichincha    A        5 2013-01-02  
1   2114           1  Quito  Pichincha    A        5 2013-01-02  
2   2112           1  Quito  Pichincha    A        5 2013-01-02  
3   2124           1  Quito  Pichincha    A        5 2013-01-02  
4   2116           1  Quito  Pichincha    A        5 2013-01-02  

(1993256, 15)


Get oil prices and combine it with the filtered dataframe

In [4]:
shared_url = f'https://drive.google.com/file/d/1j3bfQuggrmJgygdU1KbwfZSqx6Yz_6vp/view?usp=sharing'

file_id = shared_url.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

output = 'oil.parquet'
gdown.download(download_url, output, quiet=False)

oil_prices = pd.read_parquet(output)
oil_prices['date'] = pd.to_datetime(oil_prices['date'])

combined_sales_oil = pd.merge(dairy_sales_filtered, oil_prices, on='date', how='left')
print(combined_sales_oil.shape)
print()
print(combined_sales_oil.head())

Downloading...
From: https://drive.google.com/uc?id=1j3bfQuggrmJgygdU1KbwfZSqx6Yz_6vp
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\2025_03_Group_project_Supermarket\oil.parquet
100%|██████████| 11.3k/11.3k [00:00<00:00, 5.32MB/s]


(1993256, 16)

      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0        37.0         <NA>    2  2013-01  DAIRY   
4  32370         44  164088.0         8.0         <NA>    2  2013-01  DAIRY   

   class  perishable   city      state type  cluster       date  dcoilwtico  
0   2124           1  Quito  Pichincha    A        5 2013-01-02       93.14  
1   2114           1  Quito  Pichincha    A        5 2013-01-02       93.14  
2   2112           1  Quito  Pichincha    A        5 2013-01-02       93.14  
3   2124           1  Quito  Pichincha    A        5 2013-01-02       93.14  
4   2116           1  Quito  Pichincha    A        5 2013-01-02       93.14  


Get holidays national or local for quito and combine it with the filtered dataframe

In [5]:
shared_url = f'https://drive.google.com/file/d/1OHlaTspaT4hx1eCxUP8e0VBFRocrNvke/view?usp=sharing'

file_id = shared_url.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

output = 'holiday.parquet'
gdown.download(download_url, output, quiet=False)

holidays = pd.read_parquet(output)
holidays['date'] = pd.to_datetime(holidays['date'])

holidays_filtered = holidays[holidays['locale_name'].isin(['Quito', 'Ecuador'])]

print(holidays_filtered.head())
print()
print(holidays_filtered.shape)
print()

combined_sales_oil_holidays = pd.merge(combined_sales_oil, holidays_filtered, on='date', how='left')
print(combined_sales_oil_holidays.shape)
print()
print(combined_sales_oil_holidays.head())

Downloading...
From: https://drive.google.com/uc?id=1OHlaTspaT4hx1eCxUP8e0VBFRocrNvke
To: c:\Users\m.muller\Documents\Python\Projects\Supermarket CFGS\2025_03_Group_project_Supermarket\holiday.parquet
100%|██████████| 7.69k/7.69k [00:00<00:00, 7.71MB/s]


         date      type    locale locale_name  \
14 2012-08-10   Holiday  National     Ecuador   
19 2012-10-09   Holiday  National     Ecuador   
20 2012-10-12  Transfer  National     Ecuador   
21 2012-11-02   Holiday  National     Ecuador   
22 2012-11-03   Holiday  National     Ecuador   

                            description  transferred  
14        Primer Grito de Independencia        False  
19           Independencia de Guayaquil         True  
20  Traslado Independencia de Guayaquil        False  
21                      Dia de Difuntos        False  
22              Independencia de Cuenca        False  

(187, 6)

(1998943, 21)

      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0    

Get max temperature per day for Quito and add it to the dataframe

In [6]:
#get weather data for Quito
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -0.1807,  # Quito
    "longitude": -78.4678,
    "start_date": "2013-01-01",
    "end_date": "2017-08-15",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/Guayaquil"
}
response = requests.get(url, params=params)
data = response.json()

#Turn it into a dataframe with date in correct format
df_weather = pd.DataFrame(data['daily'])
df_weather['date'] = pd.to_datetime(df_weather['time'])

combined_full = pd.merge(
    combined_sales_oil_holidays,
    df_weather[['date', 'temperature_2m_max']],  # voeg hier andere kolommen toe indien gewenst
    on='date',
    how='left'
)

print(combined_full.shape)
print()
print(combined_full.head())



(1998943, 22)

      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0        37.0         <NA>    2  2013-01  DAIRY   
4  32370         44  164088.0         8.0         <NA>    2  2013-01  DAIRY   

   class  perishable   city      state type_x  cluster       date  dcoilwtico  \
0   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
1   2114           1  Quito  Pichincha      A        5 2013-01-02       93.14   
2   2112           1  Quito  Pichincha      A        5 2013-01-02       93.14   
3   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
4   2116           1  Quito  Pichincha      A        5 2013-01-02       93.14   

  type_y locale locale_

CPI data is per year and not per day (not included)

In [7]:
# CPI data from World Bank for Ecuador
data = wb.download(indicator='FP.CPI.TOTL', country='ECU', start=2013, end=2017)
print(data)


              FP.CPI.TOTL
country year             
Ecuador 2017   124.091409
        2016   123.575683
        2015   121.476252
        2014   116.841561
        2013   112.793166


C:\Users\m.muller\AppData\Local\Temp\ipykernel_16268\3781345367.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  data = wb.download(indicator='FP.CPI.TOTL', country='ECU', start=2013, end=2017)


Determine paydays and add it to the dataframe

In [8]:
combined_full['is_month_end'] = combined_full['date'].dt.is_month_end

combined_full['salary_payment'] = (combined_full['day'] == 15) | (combined_full['is_month_end'])

combined_full = combined_full.drop(columns=['is_month_end'])

print(combined_full.head())
print()
combined_full['salary_payment'].value_counts()


      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0        37.0         <NA>    2  2013-01  DAIRY   
4  32370         44  164088.0         8.0         <NA>    2  2013-01  DAIRY   

   class  perishable   city      state type_x  cluster       date  dcoilwtico  \
0   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
1   2114           1  Quito  Pichincha      A        5 2013-01-02       93.14   
2   2112           1  Quito  Pichincha      A        5 2013-01-02       93.14   
3   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
4   2116           1  Quito  Pichincha      A        5 2013-01-02       93.14   

  type_y locale locale_name descriptio

salary_payment
False    1866542
True      132401
Name: count, dtype: int64

Determine earthquakes and add it to the dataframe

In [9]:
# Aardbevingsdata als lijst van dicts
earthquake_data = [
    {'date': '2016-04-16 23:58:36', 'magnitude': 7.8},
    {'date': '2013-02-09 14:16:07', 'magnitude': 6.9},
    {'date': '2016-05-18 07:57:02', 'magnitude': 6.9},
    {'date': '2016-07-11 02:11:04', 'magnitude': 6.3},
    {'date': '2016-04-20 08:33:47', 'magnitude': 6.2},
    {'date': '2016-04-22 03:03:41', 'magnitude': 6.0},
    {'date': '2017-06-30 22:29:45', 'magnitude': 6.0}
]

df_quakes = pd.DataFrame(earthquake_data)

# Voeg een kolom toe met alleen de datum (zonder tijd)
df_quakes['datetime'] = pd.to_datetime(df_quakes['date'])
df_quakes['date'] = df_quakes['datetime'].dt.date
df_quakes['date'] = pd.to_datetime(df_quakes['date'])

df_quakes = df_quakes.drop(columns=['datetime'])

df_scope = pd.merge(combined_full, df_quakes, on='date', how='left')

print(df_scope.head())
print()
df_scope['magnitude'].value_counts()


      id  store_nbr  item_nbr  unit_sales  onpromotion  day    month family  \
0  32334         44  122095.0        14.0         <NA>    2  2013-01  DAIRY   
1  32337         44  123347.0         2.0         <NA>    2  2013-01  DAIRY   
2  32345         44  129635.0        44.0         <NA>    2  2013-01  DAIRY   
3  32362         44  158842.0        37.0         <NA>    2  2013-01  DAIRY   
4  32370         44  164088.0         8.0         <NA>    2  2013-01  DAIRY   

   class  perishable   city      state type_x  cluster       date  dcoilwtico  \
0   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
1   2114           1  Quito  Pichincha      A        5 2013-01-02       93.14   
2   2112           1  Quito  Pichincha      A        5 2013-01-02       93.14   
3   2124           1  Quito  Pichincha      A        5 2013-01-02       93.14   
4   2116           1  Quito  Pichincha      A        5 2013-01-02       93.14   

  type_y locale locale_name descriptio

magnitude
6.0    3050
6.9    2117
7.8    1498
6.3    1475
6.2    1445
Name: count, dtype: int64

Save the file as parquet and as xlsx

In [10]:
df_scope.to_parquet("df_scope.parquet", index=False)
